# Complete Machine Learning Tutorial

This notebook is a practical, beginner-to-advanced guide to Machine Learning. It explains the main ideas, gives runnable Python examples, and points you to high-quality resources for deeper study.

**How to use this notebook**

1. Run the setup cells first.
2. Read one section at a time.
3. Type small changes into every code example and rerun it.
4. After each major topic, build a mini project.
5. Keep notes about what confused you, what worked, and what you want to revise.

**Core libraries used here**: NumPy, pandas, matplotlib, seaborn, and scikit-learn.

## 1. What Is Machine Learning?

Machine Learning is a way to make computers learn patterns from data instead of writing every rule manually.

Traditional programming:

```text
rules + data -> answer
```

Machine Learning:

```text
data + answers -> learned rules/model
```

Example: Instead of manually writing rules for house prices, you give the model many houses with features such as area, rooms, location, and the actual price. The model learns the relationship and predicts prices for new houses.

### Main Types of ML

| Type | Goal | Examples |
|---|---|---|
| Supervised learning | Learn from labeled data | Regression, classification |
| Unsupervised learning | Find structure without labels | Clustering, dimensionality reduction |
| Semi-supervised learning | Use small labeled data plus large unlabeled data | Image/text labeling |
| Reinforcement learning | Learn by actions and rewards | Games, robotics, recommendation policies |

### Common ML Problems

| Problem | Output | Example |
|---|---|---|
| Regression | Number | Predict house price |
| Classification | Category | Spam or not spam |
| Clustering | Groups | Customer segments |
| Recommendation | Ranked items | Movies/products to show |
| Time series forecasting | Future values | Sales or pollution forecast |
| Anomaly detection | Unusual cases | Fraud detection |
| NLP | Text understanding/generation | Sentiment, chatbot, summarization |
| Computer vision | Image/video understanding | Tumor detection, object detection |

## 2. Environment Setup

Install the main packages if needed:

```bash
pip install numpy pandas matplotlib seaborn scikit-learn jupyter
```

Optional later:

```bash
pip install xgboost lightgbm tensorflow torch torchvision transformers shap streamlit joblib
```

For most classical ML work, start with scikit-learn. For deep learning, choose either TensorFlow/Keras or PyTorch. For NLP/LLMs, learn Hugging Face after you understand basic ML and deep learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, fetch_california_housing, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete")

## 3. ML Workflow: The Big Picture

A real ML project usually follows this flow:

1. Define the business/research problem.
2. Collect data.
3. Understand the target variable.
4. Explore data with statistics and plots.
5. Clean missing values, duplicates, outliers, and wrong types.
6. Split data into train/validation/test sets.
7. Build preprocessing steps.
8. Train a baseline model.
9. Evaluate with the right metric.
10. Improve features, model, and hyperparameters.
11. Check for overfitting, leakage, bias, and robustness.
12. Save and deploy the model.
13. Monitor performance after deployment.

### Important Vocabulary

- **Feature**: input column used by the model.
- **Target/label**: value we want to predict.
- **Training set**: data used to learn.
- **Validation set**: data used to tune choices.
- **Test set**: final untouched data used to estimate real performance.
- **Parameter**: learned by the model, such as coefficients.
- **Hyperparameter**: chosen by you, such as tree depth.
- **Loss function**: objective the model minimizes.
- **Metric**: human-friendly score used to judge performance.

## 4. Math You Need for ML

You do not need to become a mathematician first, but these ideas matter:

### Statistics

- Mean, median, variance, standard deviation
- Distribution shape
- Correlation and covariance
- Sampling and uncertainty
- Confidence intervals

### Linear Algebra

- Vectors: rows of numbers, often one data point
- Matrices: tables of numbers, often full datasets
- Dot product: weighted sum used inside many models
- Matrix multiplication: fast way to compute many predictions

### Calculus Intuition

- Gradient: direction of steepest increase
- Gradient descent: move parameters in the direction that reduces loss
- Learning rate: step size during gradient descent

### Probability

- Conditional probability
- Bayes rule
- Likelihood
- Expected value
- Entropy

Start practical, then deepen the math when models stop feeling like magic.

In [ ]:
# Small math demo: linear model prediction is a weighted sum.
features = np.array([1200, 3, 2])  # area, bedrooms, bathrooms
weights = np.array([100, 15000, 10000])
bias = 50000

predicted_price = np.dot(features, weights) + bias
print(f"Predicted price: {predicted_price:,.0f}")

## 5. Data Exploration With pandas

Before modeling, understand the data.

Check:

- Shape: rows and columns
- Data types
- Missing values
- Duplicates
- Summary statistics
- Target distribution
- Feature distributions
- Correlations
- Outliers

Good EDA prevents bad modeling decisions.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(df.shape)
display(df.head())
display(df.describe())
print(df.isna().sum())

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["MedHouseVal"], kde=True)
plt.title("Target distribution: Median House Value")
plt.show()

plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

## 6. Train/Test Split

Never evaluate your model on the same data it learned from. That gives an overly optimistic score.

Typical split:

- 70-80% training data
- 20-30% test data

For classification, use `stratify=y` so class proportions stay similar in train and test sets.

In [ ]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(X_train.shape, X_test.shape)

## 7. First Regression Model: Linear Regression

Regression predicts a number.

Examples:

- Price prediction
- Temperature prediction
- Pollution forecasting
- Sales forecasting

Linear regression learns a formula like:

```text
prediction = b0 + b1*x1 + b2*x2 + ... + bn*xn
```

It is simple, interpretable, and a strong baseline. It may underfit if the real relationship is highly nonlinear.

In [ ]:
regression_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

regression_pipeline.fit(X_train, y_train)
y_pred = regression_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2  : {r2:.3f}")

### Regression Metrics

| Metric | Meaning | Lower/Higher |
|---|---|---|
| MAE | Average absolute error | Lower is better |
| MSE | Average squared error | Lower is better |
| RMSE | Error in target units, penalizes large errors | Lower is better |
| R2 | Variance explained by model | Higher is better |

Use MAE when you want easy interpretation. Use RMSE when large errors are especially bad.

## 8. Regularization: Ridge and Lasso

Regularization reduces overfitting by penalizing overly large coefficients.

- **Ridge**: shrinks coefficients, keeps most features.
- **Lasso**: can shrink some coefficients to zero, useful for feature selection.

Regularization is controlled by `alpha`. Larger `alpha` means stronger penalty.

In [ ]:
for model in [Ridge(alpha=1.0), Lasso(alpha=0.001)]:
    pipe = Pipeline(steps=[("scaler", StandardScaler()), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
    print(type(model).__name__, "CV RMSE:", -scores.mean())

## 9. Tree-Based Regression

Decision trees split data into regions. They can model nonlinear patterns without scaling.

Pros:

- Easy to understand
- Handles nonlinear relationships
- No scaling needed

Cons:

- Can overfit badly
- Small data changes can create a different tree

Random forests train many trees and average them, usually improving stability and accuracy.

In [ ]:
models = {
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(name)
    print("MAE :", round(mean_absolute_error(y_test, preds), 3))
    print("RMSE:", round(mean_squared_error(y_test, preds, squared=False), 3))
    print("R2  :", round(r2_score(y_test, preds), 3))
    print()

## 10. Classification With Iris Dataset

Classification predicts a category.

Examples:

- Disease: yes/no
- Email: spam/not spam
- Image: cat/dog/car
- Customer: churn/not churn

The Iris dataset is a classic beginner dataset. The goal is to classify flower species from measurements.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

clf_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Classification Metrics

| Metric | Meaning | Use when |
|---|---|---|
| Accuracy | Overall correct predictions | Classes are balanced |
| Precision | Of predicted positives, how many were correct | False positives are costly |
| Recall | Of real positives, how many were found | False negatives are costly |
| F1 score | Balance of precision and recall | Data is imbalanced |
| ROC-AUC | Ranking ability across thresholds | Binary classification |

Example: In disease detection, recall is often very important because missing a sick patient can be dangerous.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 11. More Classification Models

Common models:

- Logistic Regression: strong linear baseline
- K-Nearest Neighbors: predicts from nearby examples
- Naive Bayes: fast, common for text
- Decision Tree: interpretable but can overfit
- Random Forest: robust tree ensemble
- Gradient Boosting: strong performance on tabular data
- SVM: powerful for smaller structured datasets

Always compare several models with cross-validation instead of trusting one run.

In [ ]:
classification_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

for name, model in classification_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    print(f"{name:20s} mean accuracy: {scores.mean():.3f}")

## 12. Preprocessing Real-World Data

Real datasets contain:

- Missing values
- Numeric columns
- Categorical columns
- Wrong data types
- Outliers
- Duplicates

Use `Pipeline` and `ColumnTransformer` so preprocessing is fit only on training data. This prevents data leakage.

### Data Leakage

Leakage happens when information from the test set or future becomes available during training. It makes validation scores look excellent but fails in real life.

Examples:

- Scaling before train/test split
- Filling missing values using the full dataset before split
- Using future information in time series
- Including a column that directly reveals the answer

In [ ]:
toy = pd.DataFrame({
    "age": [22, 35, np.nan, 40, 28],
    "income": [30000, 60000, 45000, np.nan, 52000],
    "city": ["Delhi", "Mumbai", "Delhi", "Pune", np.nan],
    "bought": [0, 1, 0, 1, 1],
})

X = toy.drop(columns="bought")
y = toy["bought"]

numeric_features = ["age", "income"]
categorical_features = ["city"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression()),
])

model.fit(X, y)
print("Pipeline trained")

## 13. Hyperparameter Tuning

Hyperparameters are choices made before training.

Examples:

- Tree depth
- Number of trees
- Learning rate
- Regularization strength
- Number of clusters

Use cross-validation and search methods:

- `GridSearchCV`: tries all combinations
- `RandomizedSearchCV`: samples combinations, often faster
- Bayesian optimization: smarter search, useful for expensive models

In [ ]:
iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=RANDOM_STATE, stratify=iris.target
)

rf = RandomForestClassifier(random_state=RANDOM_STATE)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 4, None],
    "min_samples_split": [2, 5],
}

search = GridSearchCV(rf, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV score:", round(search.best_score_, 3))
print("Test score:", round(search.score(X_test, y_test), 3))

## 14. Overfitting and Underfitting

### Underfitting

The model is too simple. It performs poorly on training and test data.

Fixes:

- Use a more flexible model
- Add useful features
- Train longer for neural networks
- Reduce excessive regularization

### Overfitting

The model memorizes training data. It performs well on training data but poorly on new data.

Fixes:

- Use cross-validation
- Add more data
- Simplify the model
- Regularize
- Use early stopping
- Remove leakage
- Use data augmentation for images/text

## 15. Feature Engineering

Feature engineering means creating better inputs for the model.

Examples:

- Ratios: income per family member
- Date parts: month, day, weekday, season
- Text length, word count, TF-IDF features
- Domain-specific aggregates
- Polynomial features for interactions
- Log transform for skewed numeric columns

For tabular ML, feature engineering can matter more than model choice.

In [ ]:
# Polynomial features can help linear models learn simple nonlinear relationships.
x = np.linspace(-3, 3, 100)
y = 2 * x**2 + 0.5 * x + np.random.normal(0, 1, size=100)

poly_model = Pipeline(steps=[
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear", LinearRegression()),
])

poly_model.fit(x.reshape(-1, 1), y)
y_hat = poly_model.predict(x.reshape(-1, 1))

plt.scatter(x, y, alpha=0.6, label="data")
plt.plot(x, y_hat, color="red", label="polynomial fit")
plt.legend()
plt.title("Polynomial Regression")
plt.show()

## 16. Unsupervised Learning: Clustering

Clustering finds groups when you do not have labels.

Use cases:

- Customer segmentation
- Document grouping
- Image compression
- Anomaly hints

K-Means tries to find `k` cluster centers. You must choose `k`, often using domain knowledge or the elbow method.

In [ ]:
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=RANDOM_STATE)

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init="auto")
clusters = kmeans.fit_predict(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=clusters, cmap="viridis", alpha=0.8)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", marker="X", s=200)
plt.title("K-Means Clustering")
plt.show()

## 17. Dimensionality Reduction With PCA

PCA compresses many features into fewer components while preserving as much variation as possible.

Use cases:

- Visualization
- Noise reduction
- Speeding up models
- Understanding structure

Be careful: PCA components are less interpretable than original features.

In [ ]:
iris = load_iris(as_frame=True)
scaled = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
components = pca.fit_transform(scaled)

plt.figure(figsize=(6, 5))
plt.scatter(components[:, 0], components[:, 1], c=iris.target, cmap="Set1")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris Dataset in 2 PCA Components")
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)

## 18. Imbalanced Data

Imbalanced data means one class appears much more often than another.

Example: fraud detection may have 99.5% normal transactions and 0.5% fraud.

Accuracy can be misleading. A model that predicts `not fraud` every time gets 99.5% accuracy but is useless.

Better strategies:

- Use precision, recall, F1, ROC-AUC, PR-AUC
- Use class weights
- Resample training data
- Tune prediction threshold
- Collect more minority-class examples
- Understand the business cost of each mistake

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=5,
    weights=[0.95, 0.05],
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

## 19. Model Interpretation

Interpretation answers: why did the model predict this?

Useful tools:

- Linear coefficients
- Tree feature importance
- Permutation importance
- Partial dependence plots
- SHAP values
- Error analysis by group

Interpretation is important in healthcare, finance, hiring, education, and any domain where decisions affect people.

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.plot(kind="bar", figsize=(8, 4), title="Random Forest Feature Importance")
plt.show()

display(importance)

## 20. Saving and Loading Models

After training, save the full pipeline, not only the model. The pipeline includes preprocessing, scaling, encoding, and the estimator.

In [ ]:
import joblib

iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=RANDOM_STATE, stratify=iris.target
)

final_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
final_pipeline.fit(X_train, y_train)

joblib.dump(final_pipeline, "iris_classifier_pipeline.joblib")
loaded_pipeline = joblib.load("iris_classifier_pipeline.joblib")

print("Loaded model test accuracy:", loaded_pipeline.score(X_test, y_test))

## 21. Deep Learning Overview

Deep learning uses neural networks with many layers. It is especially strong for images, audio, text, and very large datasets.

Core concepts:

- Tensor: multidimensional array
- Layer: transformation of data
- Activation: nonlinear function
- Loss: training objective
- Optimizer: updates weights
- Backpropagation: computes gradients
- Epoch: one pass over training data
- Batch: small group of examples

Popular frameworks:

- TensorFlow/Keras: beginner-friendly high-level APIs and production ecosystem
- PyTorch: popular in research and increasingly common in production

Start classical ML first for tabular data. Move to deep learning when your data type or problem needs it.

## 22. NLP, LLMs, Computer Vision, and Time Series

### NLP and LLMs

Topics to learn:

- Text cleaning
- Tokenization
- Bag of words and TF-IDF
- Word embeddings
- RNN/LSTM basics
- Transformers
- Fine-tuning pretrained models
- Prompting and retrieval-augmented generation
- Evaluation and hallucination control

### Computer Vision

Topics to learn:

- Image preprocessing
- CNNs
- Transfer learning
- Data augmentation
- Object detection
- Segmentation

### Time Series

Topics to learn:

- Trend and seasonality
- Lag features
- Rolling statistics
- Walk-forward validation
- ARIMA/SARIMA basics
- Tree models with lag features
- LSTM/GRU/Transformer forecasting

Important: time series must respect time order. Do not randomly shuffle future and past data.

## 23. Practical Project Template

Use this template for every ML project:

```text
1. Problem statement
2. Success metric
3. Dataset source and license
4. Data dictionary
5. EDA observations
6. Cleaning decisions
7. Baseline model
8. Improved model
9. Cross-validation results
10. Test-set result
11. Error analysis
12. Interpretation
13. Limitations
14. Deployment plan
15. Monitoring plan
```

### Beginner Projects

- Iris flower classification
- Titanic survival prediction
- House price regression
- Customer churn classification
- Movie review sentiment analysis

### Intermediate Projects

- Credit risk scoring
- PM2.5 forecasting
- Sales forecasting
- Product recommendation system
- Brain tumor image classifier

### Advanced Projects

- End-to-end ML app with Streamlit
- MLOps pipeline with experiment tracking
- Transformer fine-tuning
- Retrieval-augmented QA system
- Model monitoring dashboard

## 24. Best Resources

Use these in order. Do not try to study everything at once.

### Beginner ML

1. **Google Machine Learning Crash Course**: beginner-friendly explanations, videos, widgets, quizzes, and optional programming exercises.  
   https://developers.google.com/machine-learning/crash-course

2. **Kaggle Learn**: short hands-on courses for Python, pandas, intro ML, intermediate ML, feature engineering, and deep learning.  
   https://www.kaggle.com/learn

3. **scikit-learn User Guide**: the best official reference for classical ML in Python.  
   https://scikit-learn.org/stable/user_guide.html

### Math and Foundations

4. **StatQuest by Josh Starmer**: excellent visual explanations for statistics and ML.  
   https://www.youtube.com/@statquest

5. **3Blue1Brown Essence of Linear Algebra**: visual linear algebra intuition.  
   https://www.3blue1brown.com/topics/linear-algebra

6. **Mathematics for Machine Learning book**: deeper math reference.  
   https://mml-book.github.io/

### Deep Learning

7. **fast.ai Practical Deep Learning for Coders**: practical project-first deep learning.  
   https://course.fast.ai/

8. **TensorFlow Tutorials**: official TensorFlow/Keras notebook tutorials.  
   https://www.tensorflow.org/tutorials

9. **PyTorch Learn the Basics**: official PyTorch beginner workflow.  
   https://docs.pytorch.org/tutorials/beginner/basics/intro.html

### NLP and LLMs

10. **Hugging Face Course**: Transformers, datasets, tokenizers, fine-tuning, demos, and LLM topics.  
    https://huggingface.co/learn/nlp-course

### Books

11. **Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow** by Aurelien Geron.
12. **Introduction to Statistical Learning** by James, Witten, Hastie, Tibshirani, and Taylor. Free official site: https://www.statlearning.com/
13. **Pattern Recognition and Machine Learning** by Christopher Bishop, for deeper theory.

### Practice Platforms

14. Kaggle competitions and datasets: https://www.kaggle.com/
15. UCI Machine Learning Repository: https://archive.ics.uci.edu/
16. Papers with Code: https://paperswithcode.com/

### Suggested Learning Order

1. Python basics
2. NumPy and pandas
3. Visualization and EDA
4. Statistics basics
5. scikit-learn supervised learning
6. Model evaluation and validation
7. Feature engineering
8. Unsupervised learning
9. End-to-end projects
10. Deep learning
11. NLP/LLMs, CV, time series, or MLOps specialization

## 25. Interview and Revision Checklist

You should be able to explain:

- What is supervised vs unsupervised learning?
- What is train/test split?
- Why do we need validation data?
- What is overfitting?
- How do Ridge and Lasso differ?
- When is accuracy a bad metric?
- What is precision vs recall?
- What is cross-validation?
- What is data leakage?
- Why use pipelines?
- What is feature engineering?
- How does a decision tree split data?
- Why do random forests work well?
- What is gradient boosting?
- What is clustering?
- What does PCA do?
- How would you deploy a model?
- How would you monitor a model after deployment?

Final rule: A good ML engineer is not someone who only imports models. A good ML engineer understands the problem, data, metric, errors, limitations, and real-world impact.